# Infinite OCR — DOCX comparison

Scanned PDF (any script, mixed Indic + English) → **one continuous, structure-aware DOCX**.

Two methods on the SAME input, same output format, measured end-to-end:

| | Method A | Method B |
|---|---|---|
| OCR | Bodhan primary + Gemini fallback (per page) | Gemini reads page images directly (chunk-wise) |
| Structure | Nemotron VLM orders boxes + dedup; Python builds DOCX | Gemini returns typed blocks + continuation flags directly |
| Cross-page | Python regex (continuation detector) | LLM handles within chunk; Python stitches between chunks |
| Granularity | 1 call per page per stage | 1 call per CHUNK (default 3 pages) |

**Metrics**: tokens, latency, cost per stage, logged automatically for every HTTP call.

Run cells top to bottom, then jump to the Comparison section at the end.


In [ ]:
# ─── CELL 1: setup ────────────────────────────────────────────────────────
import base64, io, json, os, re, sys, time, string, datetime
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import pymupdf
import requests
from PIL import Image, ImageDraw, ImageFont

from docx import Document
from docx.shared import Pt, Inches
from html import unescape

# ── INPUT PDF ──
PDF_PATH   = Path("pdf/telugu_eng_scanned.pdf")   # change this
PAGES      = [0, 1, 2, 3, 4, 5, 6, 7, 8]          # -1 = all; or a list of 0-indexed pages e.g. [0, 2, 5]
DPI        = 200
CHUNK_SIZE = 3           # Method B: pages per Gemini call

# ── OUTPUT DIRS ──
OUT_DIR      = Path("compare_out")
METHOD_A_DIR = OUT_DIR / "A_bodhan_nemotron"
METHOD_B_DIR = OUT_DIR / "B_gemini_chunks"
CROP_DIR_A   = METHOD_A_DIR / "crops"
PAGE_DIR_A   = METHOD_A_DIR / "pages"
for d in (OUT_DIR, METHOD_A_DIR, METHOD_B_DIR, CROP_DIR_A, PAGE_DIR_A):
    d.mkdir(parents=True, exist_ok=True)

# ── CREDS ──
BODHAN_KEY     = os.environ.get("BODHAN_OCR_API_KEY")
OPENROUTER_KEY = os.environ.get("OPENROUTER_API_KEY")

# ── MODELS ──
OCR_FALLBACK_MODEL = "google/gemini-2.5-flash"              # Method A OCR fallback
ORDER_MODELS       = ["nvidia/nemotron-3.5-super-vl",       # Method A ordering (primary)
                      "google/gemini-2.5-flash"]            # Method A ordering (fallback)
METHOD_B_MODEL     = "google/gemini-2.5-flash"              # Method B chunk-wise

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
BODHAN_URL     = "https://api.bodhan.ai/v1/chat/completions"

# ── PRICING (USD per 1M tokens: input, output) ──
PRICES_USD_PER_M = {
    "google/gemini-2.5-flash":       (0.30, 2.50),
    "nvidia/nemotron-3.5-super-vl":  (0.10, 0.40),   # free on OpenRouter at time of writing
}
BODHAN_USD_PER_PAGE = 0.0024  # fill in your Bodhan per-page rate, or leave None (shown as unpriced)

# ── open PDF, resolve page selection ──
doc  = pymupdf.open(PDF_PATH)
stem = PDF_PATH.stem
total = len(doc)
page_indices = list(range(total)) if PAGES == -1 else [p for p in PAGES if 0 <= p < total]
if not page_indices:
    raise ValueError(f"no valid pages in PAGES={PAGES} for a {total}-page PDF")

print(f"PDF:    {PDF_PATH.name} ({total} pages)")
print(f"range:  processing {len(page_indices)} page(s): {page_indices}")
print(f"chunk:  Method B uses {CHUNK_SIZE} page(s) per Gemini call")
print(f"out:    {OUT_DIR}/")


PDF:    telugu_eng_scanned.pdf (4 pages)
range:  processing 4 page(s): [0, 1, 2, 3]
chunk:  Method B uses 3 page(s) per Gemini call
out:    compare_out/


In [16]:
# ─── CELL 2: metrics instrumentation (wraps requests.post) ────────────────
# Every API call to Bodhan / OpenRouter is logged with model, tokens, latency, cost.
# Call set_stage(method, stage) before each pipeline step so calls are attributed.

CALLS = []                       # one dict per HTTP call
_CTX  = {"method": "-", "stage": "-"}
_stage_t0 = {"method": "-", "stage": "-", "t": None}

def set_stage(method, stage):
    """Tag every HTTP call until the next set_stage(...) with (method, stage)."""
    global _stage_t0
    # close previous stage
    if _stage_t0["t"] is not None:
        CALLS.append({"method": _stage_t0["method"], "stage": _stage_t0["stage"],
                      "engine": "_wallclock_", "ok": True,
                      "tok_in": 0, "tok_out": 0, "cost": 0.0,
                      "seconds": time.perf_counter() - _stage_t0["t"]})
    _stage_t0.update(method=method, stage=stage, t=time.perf_counter())
    _CTX.update(method=method, stage=stage)

def reset_metrics():
    CALLS.clear(); _CTX.update(method="-", stage="-")
    _stage_t0.update(method="-", stage="-", t=None)
    print("[metrics] cleared")

def _extract(url, body_json, resp_json, dt, ok):
    model = "?"
    if isinstance(body_json, dict):
        model = body_json.get("model", model)
    if isinstance(resp_json, dict):
        model = resp_json.get("model") or model
    if model == "?" and "bodhan" in url.lower():
        model = "bodhan/indic-ocr"
    in_tok, out_tok, cost = 0, 0, None
    if isinstance(resp_json, dict):
        u = resp_json.get("usage") or {}
        in_tok  = u.get("prompt_tokens",     0) or 0
        out_tok = u.get("completion_tokens", 0) or 0
        if u.get("cost") is not None:
            cost = float(u["cost"])
    if cost is None:
        for key, (pin, pout) in PRICES_USD_PER_M.items():
            if key in model.lower():
                cost = (in_tok * pin + out_tok * pout) / 1_000_000
                break
    if cost is None and "bodhan" in model.lower() and BODHAN_USD_PER_PAGE is not None:
        cost = float(BODHAN_USD_PER_PAGE)
    return {"method": _CTX["method"], "stage": _CTX["stage"], "engine": model,
            "ok": ok, "tok_in": in_tok, "tok_out": out_tok,
            "cost": cost, "seconds": dt}

# install wrapper (idempotent — safe to re-run this cell)
if not getattr(requests.post, "_instrumented", False):
    _orig_post = requests.post

    def _tracked_post(url, *args, **kwargs):
        # ask OpenRouter to include cost in the usage block
        if "openrouter.ai" in url and isinstance(kwargs.get("json"), dict):
            kwargs["json"] = {**kwargs["json"], "usage": {"include": True}}
        t0 = time.perf_counter()
        body = kwargs.get("json")
        try:
            resp = _orig_post(url, *args, **kwargs)
        except Exception as e:
            dt = time.perf_counter() - t0
            CALLS.append(_extract(url, body, None, dt, ok=False))
            raise
        dt = time.perf_counter() - t0
        try:    data = resp.json()
        except Exception: data = None
        CALLS.append(_extract(url, body, data, dt, ok=resp.ok))
        return resp

    _tracked_post._instrumented = True
    requests.post = _tracked_post
    print("[metrics] requests.post wrapped — call set_stage(method, stage) before each step")
else:
    print("[metrics] already wrapped — this cell is idempotent")


[metrics] already wrapped — this cell is idempotent


## Shared helpers

Used by both methods: preprocessing, text sanitisers (HTML + LaTeX stripping), 
cross-page continuation detector, and the DOCX builder.

The DOCX builder takes a `page_dicts` list (one per page, each with ordered 
`regions`), plus optional `cross_joins` for sentences that span page breaks. 
Both methods produce output in this same intermediate format so the builder 
is shared.


In [22]:
# ─── CELL 4: shared helpers ──────────────────────────────────────────────
# Both methods route through these. Keep this cell focused on reusable pieces;
# method-specific logic lives in Methods A and B below.

# ── preprocess: illumination + CLAHE (BGR-native) ─────────────────────────
def _illumination_normalize(bgr, sigma=35):
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    bg = cv2.GaussianBlur(gray, (0, 0), sigma)
    norm = cv2.divide(gray, bg, scale=192).clip(0, 255).astype(np.uint8)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    lab[..., 0] = norm
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

def _clahe(bgr, clip=3.0):
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    lab[..., 0] = cv2.createCLAHE(clipLimit=clip, tileGridSize=(8, 8)).apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

def render_page(pno):
    pix = doc[pno].get_pixmap(dpi=DPI, alpha=False)
    return cv2.imdecode(np.frombuffer(pix.tobytes("png"), np.uint8), cv2.IMREAD_COLOR)

def preprocess_page(pno):
    return _clahe(_illumination_normalize(render_page(pno)))


# ── sanitisers: strip HTML tags and LaTeX math from OCR output ────────────
_HTML_TAG_RE = re.compile(r"<[^>]+>")

def _strip_html(text):
    if not text: return ""
    s = text
    s = re.sub(r"<br\s*/?>",   "\n", s, flags=re.I)
    s = re.sub(r"</tr\s*>",    "\n", s, flags=re.I)
    s = re.sub(r"</t[dh]\s*>", "\n", s, flags=re.I)
    s = _HTML_TAG_RE.sub("", s)
    s = unescape(s)
    s = re.sub(r"[ \t]+\n", "\n", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

_SUPER = str.maketrans("0123456789+-=()n", "⁰¹²³⁴⁵⁶⁷⁸⁹⁺⁻⁼⁽⁾ⁿ")
_SUB   = str.maketrans("0123456789+-=()",  "₀₁₂₃₄₅₆₇₈₉₊₋₌₍₎")
_LATEX_CMDS = {
    r"\alpha":"α", r"\beta":"β", r"\gamma":"γ", r"\delta":"δ", r"\epsilon":"ε",
    r"\theta":"θ", r"\lambda":"λ", r"\mu":"μ", r"\pi":"π", r"\rho":"ρ",
    r"\sigma":"σ", r"\tau":"τ", r"\phi":"φ", r"\omega":"ω",
    r"\Gamma":"Γ", r"\Delta":"Δ", r"\Theta":"Θ", r"\Lambda":"Λ",
    r"\Pi":"Π", r"\Sigma":"Σ", r"\Phi":"Φ", r"\Omega":"Ω",
    r"\infty":"∞", r"\times":"×", r"\cdot":"·", r"\div":"÷", r"\pm":"±",
    r"\leq":"≤", r"\geq":"≥", r"\neq":"≠", r"\approx":"≈",
    r"\rightarrow":"→", r"\partial":"∂", r"\sum":"∑", r"\int":"∫", r"\sqrt":"√",
    r"\degree":"°", r"\circ":"°",
    r"\overline":"", r"\bar":"", r"\hat":"", r"\vec":"",
    r"\text":"", r"\mathrm":"", r"\mathbf":"", r"\mathit":"",
    r"\left":"", r"\right":"",
    r"\,":" ", r"\;":" ", r"\!":"", r"\ ":" ",
}

def _sup(b):
    return b.translate(_SUPER) if b and all(c in "0123456789+-=()n" for c in b) else f"^({b})"
def _sub(b):
    return b.translate(_SUB)   if b and all(c in "0123456789+-=()"  for c in b) else f"_({b})"

def _flatten_latex(s):
    s = re.sub(r"(\d)\\frac", r"\1 \\frac", s)
    for k, v in _LATEX_CMDS.items(): s = s.replace(k, v)
    s = re.sub(r"\\frac\s*\{([^{}]*)\}\s*\{([^{}]*)\}", r"\1/\2", s)
    s = re.sub(r"\^\{([^{}]*)\}",  lambda m: _sup(m.group(1)), s)
    s = re.sub(r"\^([A-Za-z0-9])", lambda m: _sup(m.group(1)), s)
    s = re.sub(r"_\{([^{}]*)\}",   lambda m: _sub(m.group(1)), s)
    s = re.sub(r"_([A-Za-z0-9])",  lambda m: _sub(m.group(1)), s)
    s = re.sub(r"\\[A-Za-z]+", "", s)
    return s.replace("{", "").replace("}", "").strip()

def _strip_latex(text):
    if not text: return ""
    text = re.sub(r"\$\$(.+?)\$\$", lambda m: _flatten_latex(m.group(1)), text, flags=re.S)
    text = re.sub(r"\$(.+?)\$",     lambda m: _flatten_latex(m.group(1)), text, flags=re.S)
    return text

def sanitize(text):
    return _strip_latex(_strip_html(text or ""))


# ── cross-page continuation detector (shared) ─────────────────────────────
_TERMINAL_PUNCT_RE = re.compile(r"[.!?।॥:;\"')\]\}]\s*$")
_FURNITURE_RE = re.compile(
    r"^\s*("
    r"scanned\s+(by|with)\s+.+"
    r"|page\s+\d+(\s+of\s+\d+)?"
    r"|\[?\s*p\.?\s*t\.?\s*o\.?\s*\]?"
    r"|\d+-\d+-\d+"
    r"|\[\s*\]"
    r")\s*$", re.I)

_BODY_TYPES = {"paragraph", "list_item"}

def _is_real_body(region):
    if region.get("type") not in _BODY_TYPES: return False
    t = (region.get("text") or "").strip()
    if not t: return False
    return not _FURNITURE_RE.match(t)

def cross_page(page_dicts):
    """Return [(last_id, first_id), ...] to merge across page breaks."""
    joins = []
    for i in range(len(page_dicts) - 1):
        last, first = None, None
        for r in reversed(page_dicts[i]["regions"]):
            if _is_real_body(r): last = r; break
        for r in page_dicts[i+1]["regions"]:
            if _is_real_body(r): first = r; break
        if last is None or first is None: continue
        last_text = (last.get("text") or "").strip()
        if _TERMINAL_PUNCT_RE.search(last_text): continue
        joins.append((last["id"], first["id"]))
    return joins


# ── DOCX builder (shared; handles both methods' intermediate format) ──────
_FONT_NAME = "Nirmala UI"

def _apply_font(run, name=_FONT_NAME):
    run.font.name = name
    try:
        from docx.oxml.ns import qn
        from docx.oxml import OxmlElement
        rPr = run._element.get_or_add_rPr()
        rFonts = rPr.find(qn("w:rFonts"))
        if rFonts is None:
            rFonts = OxmlElement("w:rFonts"); rPr.insert(0, rFonts)
        for a in ("w:ascii", "w:hAnsi", "w:eastAsia", "w:cs"):
            rFonts.set(qn(a), name)
    except Exception:
        pass

def _add_table(doc, rows):
    if not rows: return
    ncols = max(len(r) for r in rows)
    table = doc.add_table(rows=len(rows), cols=ncols)
    try: table.style = "Light Grid Accent 1"
    except KeyError: pass
    for ri, row in enumerate(rows):
        for ci, cell in enumerate(row):
            tc = table.rows[ri].cells[ci]
            tc.text = "" if cell is None else str(cell)
            for p in tc.paragraphs:
                for run in p.runs: _apply_font(run)

def build_docx(page_dicts, cross_joins, out_path):
    """Intermediate format per region:
       {id, type, text?, image_path?, rows?, level?}
       type ∈ {title, heading, paragraph, list_item, table, picture,
               picture_description, caption, header, footer}
    """
    docx = Document()
    docx.styles["Normal"].font.name = _FONT_NAME
    docx.styles["Normal"].font.size = Pt(11)

    merge_into = {first_id: last_id for last_id, first_id in cross_joins}
    para_by_id = {}

    for page in page_dicts:
        for region in page["regions"]:
            rid   = region["id"]
            rtype = region.get("type", "paragraph")

            if rtype == "picture" and region.get("image_path"):
                try:
                    docx.add_picture(region["image_path"], width=Inches(5.5))
                except Exception as e:
                    docx.add_paragraph(f"[picture failed: {e}]")
                continue

            if rtype == "table" and region.get("rows"):
                _add_table(docx, region["rows"])
                continue

            text = sanitize(region.get("text") or "")
            if not text: continue

            if rid in merge_into and merge_into[rid] in para_by_id:
                prev = para_by_id[merge_into[rid]]
                run = prev.add_run(" " + text); _apply_font(run)
                continue

            if rtype == "title":
                p = docx.add_heading(text, level=0)
            elif rtype == "heading":
                p = docx.add_heading(text, level=min(3, max(1, region.get("level", 1))))
            elif rtype == "list_item":
                p = docx.add_paragraph(text, style="List Bullet")
            elif rtype in ("header", "footer"):
                p = docx.add_paragraph()
                r = p.add_run(text); r.italic = True; r.font.size = Pt(9)
            elif rtype == "caption":
                p = docx.add_paragraph()
                r = p.add_run(text); r.italic = True
            elif rtype == "picture_description":
                p = docx.add_paragraph()
                r = p.add_run(f"[Figure: {text}]"); r.italic = True
            else:
                p = docx.add_paragraph(text)

            for run in p.runs: _apply_font(run)
            para_by_id[rid] = p

    docx.save(str(out_path))
    return str(out_path)

def save_docx_with_fallback(page_dicts, cross_joins, path):
    try:
        return build_docx(page_dicts, cross_joins, path)
    except PermissionError:
        ts  = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        alt = path.with_name(f"{path.stem}_{ts}{path.suffix}")
        print(f"  '{path.name}' locked (open in Word?) → saving as '{alt.name}'")
        return build_docx(page_dicts, cross_joins, alt)

print("shared helpers loaded")


shared helpers loaded


## Method A — Bodhan OCR → Nemotron ordering → DOCX

Per-page pipeline, EXACTLY as in your working DOCX notebook:

1. **OCR** — Bodhan primary (specialist Indic engine); Gemini-flash fallback if Bodhan errors
2. **Dedup** — Python-side geometric dedup (IoU + text subsumption) removes Bodhan's coarse+fine box duplicates
3. **Label** — draw B0, B1, ... outside each box on the preprocessed image
4. **Order** — Nemotron 3.5 Super VL (cheap VLM) reads image + JSON, returns `{order, dropped, layout}`; Gemini fallback if it fails
5. **DOCX** — Python stitches regions in the model's order, applies cross-page continuation, writes `.docx`

The PLAN_PROMPT below is your working version — don't change it unless you want to re-tune.


In [18]:
# ─── CELL 6: Method A ─────────────────────────────────────────────────────
reset_metrics()

# ─────────────────────────────────────────────────────────────────────────
# A.1  OCR  —  Bodhan primary + Gemini fallback
# ─────────────────────────────────────────────────────────────────────────
def _b64_png(bgr):
    return base64.b64encode(cv2.imencode(".png", bgr)[1].tobytes()).decode()

def _call_bodhan(bgr, api_key, max_tokens=8192):
    payload = {
        "model": "indic-ocr",
        "messages": [{"role": "user", "content": [
            {"type": "image_url",
             "image_url": {"url": f"data:image/png;base64,{_b64_png(bgr)}"}},
        ]}],
        "max_tokens": max_tokens,
    }
    r = requests.post(BODHAN_URL,
                      headers={"Authorization": f"Bearer {api_key}"},
                      json=payload, timeout=(10, 300))
    r.raise_for_status()
    data = r.json()
    if "error" in data:
        raise RuntimeError(f"bodhan: {data['error'].get('message')}")
    data["_engine"] = "bodhan/indic-ocr"
    return data

_FALLBACK_OCR_PROMPT = (
    "You are an OCR engine. Transcribe this document page faithfully.\n"
    "Return ONLY JSON (no fences, no commentary) matching this schema:\n"
    "{\n"
    '  "blocks": [\n'
    "    {\n"
    '      "order": 1,\n'
    '      "type": "Text" | "Title" | "SectionHeader" | "List" | "Caption"\n'
    '              | "Table" | "Picture" | "PageHeader" | "PageFooter" | "Footnote",\n'
    '      "bbox_norm": [x0, y0, x1, y1] in [0,1] top-left, or null if uncertain,\n'
    '      "text": "exact content, preserve Latin+Devanagari+Telugu+... scripts"\n'
    "    }\n"
    "  ]\n"
    "}\n"
    "Rules: emit in natural reading order. Picture/signature/stamp blocks get text=\"\". "
    "Render tables as GitHub-style pipe tables. Never fabricate bbox coordinates — use null.\n"
    "\n"
    "Classify these as PageFooter / PageHeader (NOT Text) — they are page furniture, "
    "not body content, and must not be joined with the body across page breaks:\n"
    "  • scanner watermarks: \"Scanned by CamScanner\", \"Scanned with OKEN Scanner\"\n"
    "  • page codes / numbers: \"1-2-103\", \"Page 5 of 10\", \"[1]\"\n"
    "  • page-turn hints: \"[P.T.O.]\", \"Please Turn Over\", \"Continued on next page\"\n"
    "  • signatures or handwritten margin marks\n"
)

def _parse_json_object(text):
    s = text.strip()
    if s.startswith("```"):
        lines = s.splitlines()
        if lines[0].startswith("```"): lines = lines[1:]
        if lines and lines[-1].strip() == "```": lines = lines[:-1]
        s = "\n".join(lines)
    try: return json.loads(s)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", s, re.S)
        return json.loads(m.group(0)) if m else {}

def _call_openrouter_ocr(bgr, api_key, model, timeout=180):
    H, W = bgr.shape[:2]
    payload = {
        "model": model, "temperature": 0,
        "messages": [{"role": "user", "content": [
            {"type": "text", "text": _FALLBACK_OCR_PROMPT},
            {"type": "image_url",
             "image_url": {"url": f"data:image/png;base64,{_b64_png(bgr)}"}},
        ]}],
    }
    r = requests.post(OPENROUTER_URL,
                      headers={"Authorization": f"Bearer {api_key}",
                               "Content-Type": "application/json"},
                      json=payload, timeout=timeout)
    r.raise_for_status()
    data = r.json()
    if "error" in data:
        raise RuntimeError(f"openrouter: {data['error']}")
    parsed = _parse_json_object(data["choices"][0]["message"]["content"])
    # Reshape into Bodhan's envelope so downstream code is unchanged.
    blocks = []
    for i, b in enumerate(parsed.get("blocks") or [], start=1):
        bn = b.get("bbox_norm")
        bbox_px = None
        if isinstance(bn, (list, tuple)) and len(bn) == 4:
            try:
                x0, y0, x1, y1 = [max(0.0, min(1.0, float(v))) for v in bn]
                bbox_px = [round(x0*W, 1), round(y0*H, 1), round(x1*W, 1), round(y1*H, 1)]
                if bbox_px[2] <= bbox_px[0] or bbox_px[3] <= bbox_px[1]: bbox_px = None
            except Exception: bbox_px = None
        blocks.append({"order": b.get("order", i), "type": b.get("type", "Text"),
                       "text": b.get("text", "") or "", "bbox_xyxy": bbox_px})
    return {"blocks": blocks, "_engine": f"openrouter/{model}"}

def _extract_bodhan_blocks(resp):
    """Pull the blocks list out of Bodhan's response, no matter which envelope it uses."""
    # (1) directly on the top-level envelope
    if isinstance(resp.get("blocks"), list):
        return resp["blocks"]
    # (2) OpenAI-style choices[0].message.content
    content = resp.get("choices", [{}])[0].get("message", {}).get("content")
    if isinstance(content, list):          # already parsed into a list of blocks
        return content
    if isinstance(content, str):           # a JSON string we need to parse
        parsed = _parse_json_object(content)
        if isinstance(parsed, dict) and isinstance(parsed.get("blocks"), list):
            return parsed["blocks"]
        if isinstance(parsed, list):
            return parsed
    # (3) data field (some engines wrap under "data")
    if isinstance(resp.get("data"), dict) and isinstance(resp["data"].get("blocks"), list):
        return resp["data"]["blocks"]
    return []

def ocr_with_fallback(bgr):
    try:
        resp = _call_bodhan(bgr, BODHAN_KEY)
        raw = _extract_bodhan_blocks(resp)
        normalised = []
        for i, b in enumerate(raw, start=1):
            if not isinstance(b, dict): continue
            normalised.append({"order": b.get("order", i),
                               "type":  b.get("type", "Text"),
                               "text":  b.get("text", "") or "",
                               "bbox_xyxy": b.get("bbox_xyxy") or b.get("bbox")})
        # Treat "no blocks" or "all blocks empty" as a failure → trigger Gemini fallback.
        if not normalised or not any((b.get("text") or "").strip() for b in normalised):
            raise RuntimeError(f"Bodhan returned no usable blocks (got {len(normalised)}; "
                               f"top keys={list(resp.keys())[:6]})")
        return {"blocks": normalised, "_engine": "bodhan/indic-ocr"}
    except Exception as e:
        print(f"  Bodhan failed: {type(e).__name__}: {str(e)[:160]} — trying Gemini")
        if not OPENROUTER_KEY:
            raise RuntimeError("Bodhan failed and OPENROUTER_API_KEY is not set.")
        return _call_openrouter_ocr(bgr, OPENROUTER_KEY, OCR_FALLBACK_MODEL)
    

# ─────────────────────────────────────────────────────────────────────────
# A.2  Dedup  —  geometric + text subsumption (unchanged)
# ─────────────────────────────────────────────────────────────────────────
_PICTURE_TYPES = {"Picture", "Chart", "Diagram", "Image", "Figure", "picture"}

def _area(b): return max(0, b[2] - b[0]) * max(0, b[3] - b[1])
def _inter(a, b):
    x0, y0 = max(a[0], b[0]), max(a[1], b[1])
    x1, y1 = min(a[2], b[2]), min(a[3], b[3])
    return max(0, x1 - x0) * max(0, y1 - y0)
def _is_informative(block):
    if (block.get("text") or "").strip(): return True
    return block.get("type") in _PICTURE_TYPES
def _normalize_words(text):
    tbl = str.maketrans('', '', string.punctuation + "।॥")
    return {w.translate(tbl).lower() for w in (text or "").split() if w.strip()}

def dedup_boxes(blocks, iou_threshold=0.3, contain_threshold=0.95, text_subsume_threshold=0.6):
    sorted_blocks = sorted(blocks,
        key=lambda b: (_is_informative(b), _area(b.get("bbox_xyxy") or [0,0,0,0])),
        reverse=True)
    kept, removed = [], []
    for c in sorted_blocks:
        if not c.get("bbox_xyxy"): kept.append(c); continue
        a = _area(c["bbox_xyxy"]) or 1.0
        c_words = _normalize_words(c.get("text") or "")
        drop_reason = None
        for k in kept:
            if not k.get("bbox_xyxy"): continue
            inter = _inter(c["bbox_xyxy"], k["bbox_xyxy"])
            if inter / a >= contain_threshold:
                drop_reason = f"contained in B{k.get('order')} ({inter/a:.0%})"; break
            ka = _area(k["bbox_xyxy"]) or 1.0
            iou = inter / (a + ka - inter)
            if iou >= iou_threshold:
                drop_reason = f"IoU {iou:.2f} with B{k.get('order')}"; break
            if inter > 0 and c_words:
                k_words = _normalize_words(k.get("text") or "")
                if k_words:
                    ov = len(c_words & k_words) / len(c_words)
                    if ov >= text_subsume_threshold:
                        drop_reason = f"text {ov:.0%} subsumed by B{k.get('order')}"
        if drop_reason: c = {**c, "_drop_reason": drop_reason}; removed.append(c)
        else: kept.append(c)
    kept.sort(key=lambda b: b.get("order", 0))
    return kept, removed


# ─────────────────────────────────────────────────────────────────────────
# A.3  Label  —  draw B0, B1, ... OUTSIDE each box (top-left)
# ─────────────────────────────────────────────────────────────────────────
PALETTE = [(230, 25, 75), (60, 180, 75), (0, 130, 200), (245, 130, 48),
           (145, 30, 180), (0, 128, 128), (240, 50, 230), (170, 110, 40)]

def draw_labeled(img_bgr, blocks):
    pil = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(pil)
    try: font = ImageFont.truetype("arial.ttf", 24)
    except Exception: font = ImageFont.load_default()
    types = sorted({b.get("type", "?") for b in blocks})
    color_for = {t: PALETTE[i % len(PALETTE)] for i, t in enumerate(types)}
    W, H = pil.size
    for b in blocks:
        if not b.get("bbox_xyxy"): continue
        x0, y0, x1, y1 = [int(v) for v in b["bbox_xyxy"]]
        c = color_for[b.get("type", "?")]
        draw.rectangle([x0, y0, x1, y1], outline=c, width=3)
        tag = b["_label"]
        tw  = int(draw.textlength(tag, font=font)); th, pad = 28, 4
        tag_x = x0 - pad; tag_y = y0 - th - 2
        if tag_y < 0: tag_y = 0
        if tag_x < 0: tag_x = 0
        if tag_x + tw + 2*pad > W: tag_x = W - tw - 2*pad
        draw.rectangle([tag_x, tag_y, tag_x + tw + 2*pad, tag_y + th], fill=c)
        draw.text((tag_x + pad, tag_y + 2), tag, fill="white", font=font)
    return pil


# ─────────────────────────────────────────────────────────────────────────
# A.4  Ordering  —  Nemotron primary + Gemini fallback (DO NOT CHANGE)
# ─────────────────────────────────────────────────────────────────────────
PLAN_PROMPT = (
    "You are given ONE document page: the page image (every detected block is boxed "
    "and labelled B0, B1, ...) and a JSON list of those blocks. Output the reading "
    "order of the blocks.\n\n"
    "JSON fields per block: id (the label on the image), type (title, section_header, "
    "text, list_item, caption, footnote, table, picture, figure, formula, page_header, "
    "page_footer, page_number), text (truncated to start+end for long blocks). Pictures "
    "and tables may carry no text. Blocks are NOT given in reading order.\n\n"
    "STEP 1 — LAYOUT. From the image, decide the layout: single column; two or more "
    "columns; full-width blocks above/below/between columns; images or tables side by "
    "side; a mix that changes down the page. Split into horizontal zones top-to-bottom "
    "wherever a full-width block spans the page or the column structure changes.\n"
    "- single column: top to bottom.\n"
    "- 2 columns: read the LEFT column fully top-to-bottom, then the next column to its "
    "right. NEVER read across a row of a multi-column zone.\n"
    "- blocks side by side in one row (e.g. two images): left to right.\n\n"
    "STEP 2 — BASE ORDER. Zones top to bottom; within a multi-column zone, left column "
    "fully then right column fully. A caption stays with its figure/table (object then "
    "caption). A formula stays where it sits. A title or section_header stays directly "
    "before the text it introduces, in the same column.\n\n"
    "STEP 3 — HEADERS, FOOTERS, PAGE NUMBERS. These stay at their VISUAL position in "
    "the reading flow — page_header near the top, page_footer and page_number near the "
    "bottom. Footnotes come after the body of the column they belong to.\n\n"
    "STEP 4 — DEDUP. Drop a block only if BOTH:\n"
    "  (a) its bounding box overlaps another kept block's bbox by ≥40% IoU, AND\n"
    "  (b) its text is a near-verbatim subset of that block's text.\n"
    "Text similarity alone is NOT enough — keep bilingual pairs (Hindi + English "
    "translation) and MCQ options that share standard wording.\n"
    "Put only true duplicates in `dropped`.\n\n"
    "STEP 5 — CROSS-PAGE CONTINUATION. If the first body text block of this page "
    "begins mid-sentence (word fragment or lowercase), it continues from the previous "
    "page — put it FIRST, before any figure/table that was above it. If the last body "
    "text block ends cut off (mid-word or dangling), it continues to the next page — "
    "it must be the very LAST item, so put headers/footers immediately before it. A "
    "missing period alone is NOT a cut-off; on weak evidence, keep base order.\n\n"
    "MCQ / Q&A pattern: a numbered question with sub-options ((అ), (ఆ), (a), (b) etc. "
    "can be any Indic language) is one logical unit — keep its question and options "
    "together in order.\n\n"
    "Every id in `order` + `dropped` together must appear exactly once across the two "
    "lists, and only ids from the input list.\n\n"
    'Return ONLY this JSON, no prose, no code fences:\n'
    '{"layout": "<brief layout description>", '
    '"order": [...ids in reading order], '
    '"dropped": [...ids whose content is duplicated by another block]}\n\n'
    "Regions:\n"
)

def _build_order_payload(blocks, W, H, text_cap=200):
    out = []
    for b in blocks:
        bb = b.get("bbox_xyxy")
        entry = {"id": b["_label"], "type": b.get("type", "text"),
                 "bbox": [round(bb[0]/W, 3), round(bb[1]/H, 3),
                          round(bb[2]/W, 3), round(bb[3]/H, 3)] if bb else None}
        t = (b.get("text") or "").strip()
        if t:
            entry["text"] = t if len(t) <= text_cap else t[:text_cap//2] + " […] " + t[-text_cap//2:]
        out.append(entry)
    return out

def _call_ordering_model(labeled_pil, payload, api_key, model, timeout=180, extra=None):    
    buf = io.BytesIO(); labeled_pil.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode()
    body = {"model": model, "temperature": 0,
            "messages": [{"role": "user", "content": [
                {"type": "text", "text": PLAN_PROMPT + json.dumps(payload, ensure_ascii=False)},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
            ]}]}
    body.update(extra or {})
    r = requests.post(OPENROUTER_URL,
                      headers={"Authorization": f"Bearer {api_key}",
                               "Content-Type": "application/json"},
                      json=body, timeout=timeout)
    r.raise_for_status()
    data = r.json()
    if "error" in data: raise RuntimeError(f"openrouter: {data['error']}")
    return _parse_json_object(data["choices"][0]["message"]["content"])

def order_with_fallback(labeled_pil, payload):
    known = {e["id"] for e in payload}
    last_err = None
    for model in ORDER_MODELS:
        try:
            print(f"    trying {model} …")
            j = _call_ordering_model(labeled_pil, payload, OPENROUTER_KEY, model)
            order   = [i for i in (j.get("order")   or []) if i in known]
            dropped = [i for i in (j.get("dropped") or []) if i in known]
            seen = set(order) | set(dropped)
            order += [i for i in known if i not in seen]
            print(f"    ← layout: {j.get('layout','?')!r}  order={len(order)}  dropped={len(dropped)}")
            return {"order": order, "dropped": dropped, "layout": j.get("layout"), "_model": model}
        except Exception as e:
            print(f"    {model} failed: {type(e).__name__}: {str(e)[:160]}")
            last_err = e
    raise RuntimeError(f"all ordering models failed: {last_err}")


# ─────────────────────────────────────────────────────────────────────────
# A.5  Run per-page: preprocess → OCR → dedup → order → page_dict
# ─────────────────────────────────────────────────────────────────────────
BODHAN_TO_STD = {
    "Title":"title", "SectionHeader":"heading", "Text":"paragraph",
    "List":"list_item", "Caption":"caption", "Table":"paragraph",
    "Picture":"picture", "Chart":"picture", "Diagram":"picture",
    "Image":"picture", "Figure":"picture",
    "PageHeader":"header", "PageFooter":"footer", "PageNumber":"footer",
    "Header":"header", "Footer":"footer", "Footnote":"paragraph",
    "Infobox":"paragraph", "Folio":"footer",
}

ocr_responses_A, plans_A, preprocessed_A = {}, {}, {}
ocr_failures_A, order_failures_A = {}, {}

for pno in page_indices:
    print(f"\n── A · page {pno+1}/{total} ──")
    set_stage("A", "preprocess")
    img = preprocess_page(pno)
    preprocessed_A[pno] = img

    set_stage("A", "ocr")
    try:
        ocr_responses_A[pno] = ocr_with_fallback(img)
        print(f"  ocr ok: {ocr_responses_A[pno]['_engine']}  "
              f"({len(ocr_responses_A[pno]['blocks'])} blocks)")
    except Exception as e:
        ocr_failures_A[pno] = str(e); print(f"  OCR failed: {e}"); continue

    set_stage("A", "dedup+order")
    kept, removed = dedup_boxes(ocr_responses_A[pno]["blocks"])
    for i, b in enumerate(kept): b["_label"] = f"B{i}"
    print(f"  dedup: {len(ocr_responses_A[pno]['blocks'])} → kept {len(kept)}")

    labeled = draw_labeled(img, kept)
    labeled.save(PAGE_DIR_A / f"{stem}_p{pno+1}_labeled.png")

    H, W = img.shape[:2]
    payload = _build_order_payload(kept, W, H)
    try:
        plan = order_with_fallback(labeled, payload)
    except Exception as e:
        order_failures_A[pno] = str(e)
        plan = {"order": [b["_label"] for b in kept], "dropped": [],
                "layout": "fallback (ordering failed)", "_model": "fallback"}
    plans_A[pno] = {"kept": kept, **plan}

# ─────────────────────────────────────────────────────────────────────────
# A.6  Build page_dicts (crop pictures, map types, apply order)
# ─────────────────────────────────────────────────────────────────────────
set_stage("A", "build")
page_dicts_A = []
for pno in page_indices:
    if pno not in plans_A: continue
    plan = plans_A[pno]
    idmap = {b["_label"]: b for b in plan["kept"]}
    pil = Image.fromarray(cv2.cvtColor(preprocessed_A[pno], cv2.COLOR_BGR2RGB))
    regions = []
    for i, rid in enumerate(plan["order"], start=1):
        if rid not in idmap: continue
        b = idmap[rid]
        std = BODHAN_TO_STD.get(b.get("type"), "paragraph")
        reg = {"id": f"A_p{pno+1}_r{i}", "type": std}
        if std == "picture" and b.get("bbox_xyxy"):
            x0, y0, x1, y1 = [int(v) for v in b["bbox_xyxy"]]
            crop_path = CROP_DIR_A / f"{stem}_p{pno+1}_{rid}.png"
            crop_path.parent.mkdir(parents=True, exist_ok=True)
            pil.crop((x0, y0, x1, y1)).save(crop_path)
            reg["image_path"] = str(crop_path)
        else:
            reg["text"] = b.get("text") or ""
        regions.append(reg)
    page_dicts_A.append({"page": pno + 1, "regions": regions})

cross_A = cross_page(page_dicts_A)
print(f"\nA · page_dicts: {len(page_dicts_A)}  ·  cross-page joins: {len(cross_A)}")

docx_A_path = METHOD_A_DIR / f"{stem}__A.docx"
final_A = save_docx_with_fallback(page_dicts_A, cross_A, docx_A_path)
print(f"A · saved {final_A}")
set_stage("-", "-")


[metrics] cleared

── A · page 1/4 ──
  ocr ok: bodhan/indic-ocr  (12 blocks)
  dedup: 12 → kept 11
    trying nvidia/nemotron-3.5-super-vl …
    ← layout: 'Single column layout with header blocks at the top, followed by section headers and question blocks, and footer information at the bottom right.'  order=10  dropped=1

── A · page 2/4 ──
  ocr ok: bodhan/indic-ocr  (7 blocks)
  dedup: 7 → kept 7
    trying nvidia/nemotron-3.5-super-vl …
    ← layout: 'Single column layout with questions 11-13 stacked vertically at the top, followed by a handwritten block and footer elements at the bottom right.'  order=7  dropped=0

── A · page 3/4 ──
  ocr ok: bodhan/indic-ocr  (17 blocks)
  dedup: 17 → kept 14
    trying nvidia/nemotron-3.5-super-vl …
    ← layout: 'Single column layout with header blocks at the top, followed by instructional text and questions, and footer blocks at the bottom right.'  order=12  dropped=2

── A · page 4/4 ──
  ocr ok: bodhan/indic-ocr  (11 blocks)
  dedup: 11 → k

## Method B — Gemini chunk-wise → DOCX

Chunk-wise pipeline, new design:

1. **Chunk** — pages split into groups of `CHUNK_SIZE` (default 3)
2. **One Gemini call per chunk** — all N page-images in one request. The prompt asks for a strict JSON output: typed blocks per page (title / heading / paragraph / list_item / table / picture_description) plus `chunk_starts_continuation` / `chunk_ends_continuation` flags
3. **Within a chunk**: Gemini handles cross-page continuations naturally (sentences, lists, tables that span pages inside the chunk flow as one)
4. **Between chunks**: Python stitches using the continuation flags; if chunk K ended mid-sentence AND chunk K+1 starts mid-sentence, merge last block of K with first block of K+1
5. **DOCX** — same shared builder as Method A

No separate OCR step, no bbox labelling, no dedup — Gemini does it all in one pass per chunk.


In [20]:
# ─── CELL 8: Method B ─────────────────────────────────────────────────────
# NOTE: do NOT call reset_metrics() here — we want A's metrics kept alongside B's.

METHOD_B_PROMPT_HEAD = """You are a meticulous document transcriber. You are given {n} consecutive pages ({first}–{last}) of ONE scanned document, as images in order.

Transcribe ALL content on these pages in reading order, into a structured JSON format described below. The content may be in English, Hindi, Telugu, Kannada, or any Indic script — preserve every script EXACTLY as printed.

FIDELITY — highest priority
- Transcribe EXACTLY what is printed. Never translate, summarize, "correct" spelling, normalize punctuation, merge or reorder questions.
- Preserve multilingual text (UTF-8), the document's own numbering ("1.", "(a)", "(क)", "प्रश्न 1"), marks notation ("[5 marks]", "3×2=6"), times, "OR" / "अथवा" lines, and fill-in-blank dot sequences (as literal dots).
- Text you cannot read: write "[illegible]". Never guess.

PAGE FURNITURE — OMIT entirely (no text, no placeholder)
- Scanner watermarks ("Scanned with OKEN Scanner", "Scanned by CamScanner")
- Page numbers, "[P.T.O.]", "Continued on next page", exam folios like "1-2-103"
- Repeating running headers/footers (college name shown ONLY on the first page is NOT furniture; the same text on EVERY page is)

READING ORDER
- Multi-column: read the LEFT column fully top-to-bottom, then the next column.
- A caption stays with its figure/table.
- A heading stays directly before the text it introduces.

CROSS-PAGE CONTINUATIONS (within this chunk)
- If a sentence, list, or table spans two pages inside this chunk, produce it as ONE natural flow — do NOT restart, do NOT label the page break. The reader should see the complete sentence/list/table.
- Set `chunk_starts_continuation` = true ONLY if the FIRST block of page {first} obviously starts mid-sentence (lowercase word, fragment) from a page not shown.
- Set `chunk_ends_continuation` = true ONLY if the LAST block of page {last} is clearly cut off mid-sentence and continues on a page not shown.

PICTURES
- You cannot crop images. For diagrams, charts, stamps, signatures: use type "picture_description" with a short factual description (what it depicts, visible labels). Do not transcribe stamps or signatures as text.

OUTPUT — strict JSON, no prose, no code fences. Schema:

{{
  "pages": [
    {{
      "page_num": <int, matches the printed page number if visible, else the ordinal>,
      "blocks": [
        {{"type": "title", "text": "..."}},
        {{"type": "heading", "level": 1|2|3, "text": "..."}},
        {{"type": "paragraph", "text": "..."}},
        {{"type": "list_item", "text": "..."}},
        {{"type": "table", "rows": [["cell", "cell"], ["cell", "cell"]]}},
        {{"type": "picture_description", "text": "..."}}
      ]
    }}
  ],
  "chunk_starts_continuation": false,
  "chunk_ends_continuation": false
}}

Return the JSON only.
"""

def _pil_data_uri(bgr):
    return f"data:image/png;base64,{_b64_png(bgr)}"

def _call_B_chunk(pnos, images_bgr, model, timeout=420):
    """One Gemini call per chunk. Returns parsed dict."""
    first, last, n = pnos[0] + 1, pnos[-1] + 1, len(pnos)
    text = METHOD_B_PROMPT_HEAD.format(n=n, first=first, last=last)
    content = [{"type": "text", "text": text}]
    for img in images_bgr:
        content.append({"type": "image_url", "image_url": {"url": _pil_data_uri(img)}})
    body = {"model": model, "temperature": 0,
            "messages": [{"role": "user", "content": content}]}
    r = requests.post(OPENROUTER_URL,
                      headers={"Authorization": f"Bearer {OPENROUTER_KEY}",
                               "Content-Type": "application/json"},
                      json=body, timeout=timeout)
    r.raise_for_status()
    data = r.json()
    if "error" in data: raise RuntimeError(f"openrouter: {data['error']}")
    parsed = _parse_json_object(data["choices"][0]["message"]["content"])
    if not isinstance(parsed.get("pages"), list):
        raise ValueError(f"bad JSON shape (no 'pages' list): {str(parsed)[:200]}")
    return parsed

def _fallback_B_page(pno):
    """When the chunk LLM fails on a single page even after splitting."""
    return {"page_num": pno + 1,
            "blocks": [{"type": "paragraph", "text": f"[Method B could not transcribe page {pno+1}]"}]}

def solve_B_chunk(pnos, images_bgr):
    """Try the chunk; on failure, split in half; down to single pages, then fallback."""
    try:
        print(f"  [B] Gemini on pages {pnos[0]+1}–{pnos[-1]+1} …")
        r = _call_B_chunk(pnos, images_bgr, METHOD_B_MODEL)
        print(f"  ← pages={len(r.get('pages', []))}  cont s/e: "
              f"{r.get('chunk_starts_continuation', False)}/{r.get('chunk_ends_continuation', False)}")
        return r, {}
    except Exception as e:
        print(f"  B chunk failed: {type(e).__name__}: {str(e)[:180]}")
        if len(pnos) == 1:
            return {"pages": [_fallback_B_page(pnos[0])],
                    "chunk_starts_continuation": False,
                    "chunk_ends_continuation": False}, {pnos[0]: str(e)}
        mid = len(pnos) // 2
        print(f"  ⚠ splitting chunk {pnos[0]+1}–{pnos[-1]+1}")
        left,  f_l = solve_B_chunk(pnos[:mid], images_bgr[:mid])
        right, f_r = solve_B_chunk(pnos[mid:], images_bgr[mid:])
        merged = {"pages": left["pages"] + right["pages"],
                  "chunk_starts_continuation": left["chunk_starts_continuation"],
                  "chunk_ends_continuation":   right["chunk_ends_continuation"]}
        return merged, {**f_l, **f_r}


# ─────────────────────────────────────────────────────────────────────────
# B.1  Preprocess each page once (same CLAHE as Method A — fair comparison)
# ─────────────────────────────────────────────────────────────────────────
set_stage("B", "preprocess")
preprocessed_B = {}
for pno in page_indices:
    # If Method A already preprocessed this page, reuse it (same image).
    img_cached = preprocessed_A.get(pno)
    preprocessed_B[pno] = img_cached if img_cached is not None else preprocess_page(pno)


# ─────────────────────────────────────────────────────────────────────────
# B.2  Chunk pages, call Gemini per chunk
# ─────────────────────────────────────────────────────────────────────────
chunks_B = [page_indices[i:i + CHUNK_SIZE] for i in range(0, len(page_indices), CHUNK_SIZE)]
print(f"\nB · {len(page_indices)} page(s) → {len(chunks_B)} chunk(s) of up to {CHUNK_SIZE}")

set_stage("B", "chunked_llm")
chunk_results_B, chunk_failures_B = [], {}
for ci, chunk in enumerate(chunks_B, 1):
    print(f"\n── B · chunk {ci}/{len(chunks_B)}: pages {chunk[0]+1}–{chunk[-1]+1} ──")
    images = [preprocessed_B[p] for p in chunk]
    result, fails = solve_B_chunk(chunk, images)
    chunk_results_B.append({"pnos": chunk, **result})
    chunk_failures_B.update(fails)


# ─────────────────────────────────────────────────────────────────────────
# B.3  Flatten chunks → page_dicts, handle BETWEEN-chunk continuation
# ─────────────────────────────────────────────────────────────────────────
set_stage("B", "build")
page_dicts_B = []
cross_B = []
prev_chunk_last_rid = None
prev_chunk_ends_cont = False

for ci, result in enumerate(chunk_results_B):
    chunk_pnos = result["pnos"]
    pages = result.get("pages") or []
    # Pad / trim to the chunk size if the model under/over-returned
    for local_i, pno in enumerate(chunk_pnos):
        if local_i < len(pages):
            page_blocks = pages[local_i].get("blocks") or []
        else:
            page_blocks = [{"type": "paragraph", "text": f"[page {pno+1}: no content returned]"}]
        regions = []
        for bi, b in enumerate(page_blocks, start=1):
            if not isinstance(b, dict): continue
            btype = b.get("type", "paragraph")
            reg = {"id": f"B_c{ci+1}_p{pno+1}_r{bi}", "type": btype}
            if btype == "table":
                reg["rows"] = b.get("rows") or []
            else:
                reg["text"] = b.get("text") or ""
                if btype == "heading":
                    reg["level"] = int(b.get("level", 1) or 1)
            regions.append(reg)
        page_dicts_B.append({"page": pno + 1, "regions": regions})

    # BETWEEN-chunk continuation: if the previous chunk ENDED mid-sentence AND
    # THIS chunk STARTS mid-sentence, merge the last body of previous into first body of this.
    if ci > 0 and prev_chunk_ends_cont and result.get("chunk_starts_continuation") and prev_chunk_last_rid:
        # find the first real-body region of this chunk's pages
        for pd in page_dicts_B[-len(chunk_pnos):]:
            found = False
            for r in pd["regions"]:
                if _is_real_body({"type": r["type"], "text": r.get("text", "")}):
                    cross_B.append((prev_chunk_last_rid, r["id"]))
                    found = True; break
            if found: break

    # remember the last body region of THIS chunk for the next iteration
    prev_chunk_ends_cont = bool(result.get("chunk_ends_continuation"))
    last_rid = None
    for pd in reversed(page_dicts_B[-len(chunk_pnos):]):
        for r in reversed(pd["regions"]):
            if _is_real_body({"type": r["type"], "text": r.get("text", "")}):
                last_rid = r["id"]; break
        if last_rid: break
    prev_chunk_last_rid = last_rid

# ALSO run Python's cross-page detector WITHIN chunks for pages where the model
# didn't flow a continuation naturally (safety net). Only adds joins; never undoes any.
extra = cross_page(page_dicts_B)
# merge, dedup
existing = {j for j in cross_B}
for j in extra:
    if j not in existing: cross_B.append(j)

print(f"\nB · page_dicts: {len(page_dicts_B)}  ·  cross-page joins: {len(cross_B)}  "
      f"({sum(1 for _ in chunk_results_B if _.get('chunk_ends_continuation'))} chunks ended mid-sentence)")

docx_B_path = METHOD_B_DIR / f"{stem}__B.docx"
final_B = save_docx_with_fallback(page_dicts_B, cross_B, docx_B_path)
print(f"B · saved {final_B}")
set_stage("-", "-")



B · 4 page(s) → 2 chunk(s) of up to 3

── B · chunk 1/2: pages 1–3 ──
  [B] Gemini on pages 1–3 …
  ← pages=3  cont s/e: False/False

── B · chunk 2/2: pages 4–4 ──
  [B] Gemini on pages 4–4 …
  ← pages=1  cont s/e: False/False

B · page_dicts: 4  ·  cross-page joins: 0  (0 chunks ended mid-sentence)
B · saved compare_out\B_gemini_chunks\telugu_eng_scanned__B.docx


## Comparison — tokens · latency · cost · outcome

Breaks down the metrics logged during both runs:

- **Per stage, per engine**: calls, tokens in/out, API time, USD cost
- **Per method total**: wall-clock, cost per page, blended $/1M tokens
- **Outcome**: file size, paragraph count, image count, table count

The OCR costs should be roughly equal between methods if preprocessing is 
the same. The real signal is in `dedup+order` (A) vs `chunked_llm` (B) — 
same job, different strategy.


In [24]:
# ─── CELL 10: comparison report ──────────────────────────────────────────
_NAMES = {
    "A": "A · Bodhan OCR (+ Gemini fallback) → Nemotron ordering → DOCX",
    "B": f"B · Gemini chunk-wise ({CHUNK_SIZE} pages/call) → DOCX",
}

def _stage_order(method):
    seen = []
    for c in CALLS:
        if c["method"] == method and c["stage"] not in seen:
            seen.append(c["stage"])
    return seen

def method_report(method):
    calls = [c for c in CALLS if c["method"] == method and c["engine"] != "_wallclock_"]
    wall  = [c for c in CALLS if c["method"] == method and c["engine"] == "_wallclock_"]
    if not calls and not wall:
        print(f"\n[{method}] no calls logged"); return None

    print(f"\n{'═'*98}")
    print(f" METHOD {_NAMES[method]}")
    print(f"{'═'*98}")
    print(f" {'stage':<16}{'engine':<32}{'calls ok/fail':>14}{'tok in':>10}{'tok out':>10}"
          f"{'api s':>8}{'cost $':>10}")
    print(" " + "─"*96)

    tot = dict(ok=0, fail=0, i=0, o=0, s=0.0, cost=0.0, unpriced=0)
    for st in _stage_order(method):
        engines = sorted({c["engine"] for c in calls if c["stage"] == st})
        for en in engines:
            cs = [c for c in calls if c["stage"] == st and c["engine"] == en]
            ok = [c for c in cs if c["ok"]]
            cost = sum((c["cost"] or 0) for c in ok)
            unp  = sum(1 for c in ok if c["cost"] is None)
            vals = (len(ok), len(cs) - len(ok), sum(c["tok_in"] for c in ok),
                    sum(c["tok_out"] for c in ok),
                    sum(c["seconds"] for c in cs), cost)
            flag = " *" if unp else ""
            print(f" {st[:16]:<16}{en[:31]:<32}{f'{vals[0]}/{vals[1]}':>14}"
                  f"{vals[2]:>10,}{vals[3]:>10,}{vals[4]:>8.1f}{vals[5]:>10.4f}{flag}")
            tot["ok"] += vals[0]; tot["fail"] += vals[1]
            tot["i"]  += vals[2]; tot["o"] += vals[3]
            tot["s"]  += vals[4]; tot["cost"] += vals[5]
            tot["unpriced"] += unp

    wall_by_stage = defaultdict(float)
    for c in wall:
        wall_by_stage[c["stage"]] += c["seconds"]
    print(" " + "─"*96)
    print(" wall-clock by stage (s): " +
          " · ".join(f"{k} {v:.1f}" for k, v in wall_by_stage.items()))
    total_wall = sum(wall_by_stage.values())
    n_pages = len(page_indices)
    per_page_cost = tot["cost"] / max(n_pages, 1)
    print(f" TOTAL  wall {total_wall:.1f}s ({total_wall/max(n_pages,1):.1f}s/page)  ·  "
          f"API {tot['s']:.1f}s  ·  tokens in {tot['i']:,} / out {tot['o']:,}  ·  "
          f"cost ${tot['cost']:.4f} (${per_page_cost:.4f}/page)")
    if tot["unpriced"]:
        print(f" *  {tot['unpriced']} call(s) unpriced (Bodhan). Set BODHAN_USD_PER_PAGE in Cell 1.")
    return {"wall_s": total_wall, "api_s": tot["s"],
            "tok_in": tot["i"], "tok_out": tot["o"],
            "cost": tot["cost"], "unpriced": tot["unpriced"]}

def file_outcome(method, path):
    if not Path(path).exists():
        return {"path": None, "size_kb": 0, "paragraphs": 0, "tables": 0, "pictures": 0}
    d = Document(path)
    paras   = sum(1 for p in d.paragraphs if (p.text or "").strip())
    tables  = len(d.tables)
    pictures = 0
    for sh in d.inline_shapes:
        try: pictures += 1
        except Exception: pass
    return {"path": str(path),
            "size_kb": round(Path(path).stat().st_size / 1024, 1),
            "paragraphs": paras, "tables": tables, "pictures": pictures}

# per-method breakdowns
tot_A = method_report("A")
tot_B = method_report("B")

out_A = file_outcome("A", final_A if 'final_A' in dir() else "")
out_B = file_outcome("B", final_B if 'final_B' in dir() else "")

print(f"\n{'═'*98}")
print(f" SIDE-BY-SIDE  (pages processed: {len(page_indices)})")
print(f"{'═'*98}")
print(f" {'metric':<28}{'Method A':>30}{'Method B':>30}")
print(" " + "─"*88)
def _row(label, a, b, fmt=str):
    print(f" {label:<28}{fmt(a):>30}{fmt(b):>30}")

if tot_A and tot_B:
    _row("wall-clock (s)",   tot_A["wall_s"], tot_B["wall_s"], lambda x: f"{x:.1f}")
    _row("API time (s)",     tot_A["api_s"],  tot_B["api_s"],  lambda x: f"{x:.1f}")
    _row("tokens in",        tot_A["tok_in"], tot_B["tok_in"], lambda x: f"{x:,}")
    _row("tokens out",       tot_A["tok_out"],tot_B["tok_out"],lambda x: f"{x:,}")
    _row("cost (USD)",       tot_A["cost"],   tot_B["cost"],   lambda x: f"${x:.4f}")
    _row("cost per page",    tot_A["cost"] / len(page_indices),
                             tot_B["cost"] / len(page_indices),
                             lambda x: f"${x:.4f}")

_row("DOCX size (KB)",     out_A["size_kb"],    out_B["size_kb"])
_row("paragraphs",         out_A["paragraphs"], out_B["paragraphs"])
_row("tables",             out_A["tables"],     out_B["tables"])
_row("pictures",           out_A["pictures"],   out_B["pictures"])
_row("file",               out_A["path"] or "—", out_B["path"] or "—")

print(f"\n{'═'*98}")
print(" Open both DOCXs in Word for a visual spot-check — the real quality signal.")
print(f"{'═'*98}")



══════════════════════════════════════════════════════════════════════════════════════════════════
 METHOD A · Bodhan OCR (+ Gemini fallback) → Nemotron ordering → DOCX
══════════════════════════════════════════════════════════════════════════════════════════════════
 stage           engine                           calls ok/fail    tok in   tok out   api s    cost $
 ────────────────────────────────────────────────────────────────────────────────────────────────
 ocr             indic-ocr                                  4/0     9,358     1,417    12.9    0.0000 *
 dedup+order     nvidia/nemotron-3.5-super-vl               4/0    19,361    37,674    74.5    0.0170
 ────────────────────────────────────────────────────────────────────────────────────────────────
 wall-clock by stage (s): preprocess 2.9 · ocr 13.4 · dedup+order 78.6 · build 0.1
 TOTAL  wall 95.0s (23.8s/page)  ·  API 87.3s  ·  tokens in 28,719 / out 39,091  ·  cost $0.0170 ($0.0043/page)
 *  4 call(s) unpriced (Bodhan).

In [25]:
# ─── CELL 11: formatter sweep — 3 small models on Method A's DOCX ────────
# Reads Method A's output, feeds the raw text to each small model with the
# same prompt, writes one DOCX per model into METHOD_A_DIR. Compares cost +
# latency at the end. Visual quality: open all three in Word and judge.

FORMAT_MODELS = [
    "meta-llama/llama-3.2-3b-instruct",      # cheapest — risky on Indic script
    "qwen/qwen-2.5-7b-instruct",             # slightly bigger
    "google/gemini-flash-1.5-8b",            # best Indic coverage at this price
]
FORMAT_CHUNK_WORDS = 1500  # per LLM call; chunks split on sentence boundaries

# ── one small-model prompt, used by every model ──────────────────────────
FORMAT_PROMPT = """You are a document-formatting assistant. The INPUT is OCR text from a scanned
exam paper. Content is in English and Indian scripts (Telugu, Devanagari, etc.) — preserve
every script EXACTLY, UTF-8. Your job: reformat into clean paragraph structure.

HARD RULES — your output MUST be a word-for-word superset of the input content:
- DO NOT translate, paraphrase, or correct spelling.
- DO NOT complete sentences, add words, or invent content.
- DO NOT change numbers, question labels, or option markers.
- Content you MAY remove (page furniture — these only):
    * scanner watermarks like "Scanned by CamScanner", "Scanned with OKEN Scanner"
    * page codes like "1-2-103", "1-6-103", "Page 5 of 10"
    * "[P.T.O.]", "Please Turn Over", "Continued on next page"
    * handwritten signatures / margin marks like "J. H. Polk"

STRUCTURE RULES — how to lay the content out:
- Split any line that contains multiple numbered items ("1. ... 2. ... 3. ...") into separate paragraphs, one per item.
- For an exam question with alternative options ((అ)/(ఆ), (a)/(b), (क)/(ख)): put the stem on one line, each option on its own line prefixed with `- `.
- A separator word on its own ("లేదా", "OR", "अथवा", "లేదా") → write it on its own line as `---` (horizontal rule).
- A section header like "విభాగం - ఎ", "PART-A", "SECTION B", "भाग - अ" → prefix with `## `.
- A document title (college name, exam name) → prefix with `# `.
- Marks notation like "(మార్కులు : 5 × 5 = 25)" or "[5 Marks]" → put on its OWN line, enclose in asterisks for italics: `*(మార్కులు : 5 × 5 = 25)*`.
- Separate paragraphs with ONE blank line.
- NEVER use code fences. NEVER add any preamble like "Here is the formatted text:". Just emit the reformatted content.

INPUT:
"""


# ── helpers ─────────────────────────────────────────────────────────────
def _chunk_by_words(text, n_words):
    """Split text into ~n_words chunks, breaking only on sentence terminators."""
    words = text.split()
    out, cur, cur_n = [], [], 0
    for w in words:
        cur.append(w); cur_n += 1
        if cur_n >= n_words and w.endswith((".", "?", "!", "।", "॥", ":", ";")):
            out.append(" ".join(cur)); cur, cur_n = [], 0
    if cur:
        out.append(" ".join(cur))
    return out

def _call_formatter(text_chunk, model, timeout=180):
    body = {"model": model, "temperature": 0,
            "messages": [{"role": "user", "content": FORMAT_PROMPT + text_chunk}]}
    r = requests.post(OPENROUTER_URL,
                      headers={"Authorization": f"Bearer {OPENROUTER_KEY}",
                               "Content-Type": "application/json"},
                      json=body, timeout=timeout)
    r.raise_for_status()
    data = r.json()
    if "error" in data: raise RuntimeError(f"openrouter: {data['error']}")
    return data["choices"][0]["message"]["content"].strip()

def _formatted_to_docx(formatted_text, out_path):
    """Build DOCX from the markdown-like format the formatter emits."""
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    docx = Document()
    docx.styles["Normal"].font.name = _FONT_NAME
    docx.styles["Normal"].font.size = Pt(11)

    for raw_para in re.split(r"\n\s*\n", formatted_text.strip()):
        lines = [l.rstrip() for l in raw_para.split("\n") if l.strip()]
        if not lines: continue

        # separator (horizontal rule)
        if len(lines) == 1 and lines[0].strip() in ("---", "***", "___"):
            p = docx.add_paragraph("─" * 20)
            p.alignment = WD_ALIGN_PARAGRAPH.CENTER
            for run in p.runs: _apply_font(run)
            continue

        # multi-line bullet block
        if all(l.lstrip().startswith("- ") for l in lines):
            for l in lines:
                p = docx.add_paragraph(l.lstrip()[2:], style="List Bullet")
                for run in p.runs: _apply_font(run)
            continue

        # per-line decision for single-line blocks or mixed
        for ln in lines:
            stripped = ln.lstrip()
            if stripped.startswith("### "):
                p = docx.add_heading(stripped[4:], level=3)
            elif stripped.startswith("## "):
                p = docx.add_heading(stripped[3:], level=2)
            elif stripped.startswith("# "):
                p = docx.add_heading(stripped[2:], level=1)
            elif stripped.startswith("- "):
                p = docx.add_paragraph(stripped[2:], style="List Bullet")
            elif stripped.startswith("*") and stripped.endswith("*") and len(stripped) > 2:
                p = docx.add_paragraph()
                run = p.add_run(stripped.strip("*"))
                run.italic = True
                run.font.size = Pt(10)
            else:
                p = docx.add_paragraph(ln)
            for run in p.runs: _apply_font(run)

    docx.save(str(out_path))
    return str(out_path)


# ── load Method A's DOCX as raw text ────────────────────────────────────
if "final_A" not in dir() or not Path(final_A).exists():
    raise RuntimeError("Method A DOCX not found — run Method A first")

src_doc  = Document(final_A)
raw_text = "\n\n".join(p.text for p in src_doc.paragraphs if (p.text or "").strip())
in_words = len(raw_text.split())
chunks   = _chunk_by_words(raw_text, FORMAT_CHUNK_WORDS)
print(f"input: {len(raw_text):,} chars · {in_words:,} words · {len(chunks)} chunk(s)")

# ── run each model, save a DOCX per model ───────────────────────────────
formatter_results = {}

for model in FORMAT_MODELS:
    short = model.split("/")[-1].replace(":", "_")
    stage_name = f"fmt_{short[:14]}"
    set_stage("A_format", stage_name)
    print(f"\n── {model} ──")
    pieces, n_failed = [], 0
    for ci, ch in enumerate(chunks, 1):
        try:
            out = _call_formatter(ch, model)
            # strict safeguard: output must have at least 90% as many words as input
            if len(out.split()) < 0.90 * len(ch.split()):
                raise RuntimeError(f"output has only {len(out.split())} words vs input {len(ch.split())} — likely truncated/dropped content")
            pieces.append(out)
            print(f"  chunk {ci}/{len(chunks)}: {len(out.split())} words ← {len(ch.split())} words")
        except Exception as e:
            print(f"  chunk {ci}/{len(chunks)} FAILED: {type(e).__name__}: {str(e)[:160]}")
            print(f"    keeping raw chunk (no formatting)")
            pieces.append(ch); n_failed += 1

    formatted = "\n\n".join(pieces)
    out_path  = METHOD_A_DIR / f"{stem}__A_fmt_{short}.docx"
    final_fmt = _formatted_to_docx(formatted, out_path)

    # cost + latency from the stage we just ran
    stage_calls = [c for c in CALLS if c["stage"] == stage_name and c["engine"] != "_wallclock_"]
    cost = sum((c["cost"] or 0) for c in stage_calls)
    tin  = sum(c["tok_in"]  for c in stage_calls)
    tout = sum(c["tok_out"] for c in stage_calls)
    sec  = sum(c["seconds"] for c in stage_calls)
    out_words = len(formatted.split())
    formatter_results[model] = {
        "docx": final_fmt, "cost": cost, "sec": sec,
        "tok_in": tin, "tok_out": tout, "failed": n_failed, "out_words": out_words,
    }
    print(f"  saved {Path(final_fmt).name}")
    print(f"  {len(stage_calls)} call(s), {tin:,}/{tout:,} tok, {sec:.1f}s, ${cost:.5f}, "
          f"failed {n_failed}, output {out_words} words")

set_stage("-", "-")

# ── comparison table ────────────────────────────────────────────────────
print(f"\n{'═'*98}")
print(f" FORMATTER SWEEP  (input: {in_words:,} words)")
print(f"{'═'*98}")
print(f" {'model':<40}{'words out':>12}{'tok in':>10}{'tok out':>10}{'sec':>8}{'cost $':>12}{'fail':>6}")
print(" " + "─" * 96)
for model, r in formatter_results.items():
    print(f" {model[:40]:<40}{r['out_words']:>12,}{r['tok_in']:>10,}{r['tok_out']:>10,}"
          f"{r['sec']:>8.1f}{r['cost']:>12.5f}{r['failed']:>6}")

print(f"\nOpen all three DOCXs in Word side-by-side to judge quality:")
for model, r in formatter_results.items():
    print(f"  {model.split('/')[-1]:<32} → {r['docx']}")

input: 3,216 chars · 465 words · 1 chunk(s)

── meta-llama/llama-3.2-3b-instruct ──
  chunk 1/1: 6370 words ← 465 words
  saved telugu_eng_scanned__A_fmt_llama-3.2-3b-instruct.docx
  1 call(s), 5,065/26,401 tok, 167.7s, $0.00910, failed 0, output 6370 words

── qwen/qwen-2.5-7b-instruct ──
  chunk 1/1: 430 words ← 465 words
  saved telugu_eng_scanned__A_fmt_qwen-2.5-7b-instruct.docx
  1 call(s), 4,561/3,828 tok, 31.8s, $0.00122, failed 0, output 430 words

── google/gemini-flash-1.5-8b ──
  chunk 1/1 FAILED: HTTPError: 404 Client Error: Not Found for url: https://openrouter.ai/api/v1/chat/completions
    keeping raw chunk (no formatting)
  saved telugu_eng_scanned__A_fmt_gemini-flash-1.5-8b.docx
  1 call(s), 0/0 tok, 0.0s, $0.00000, failed 1, output 465 words

══════════════════════════════════════════════════════════════════════════════════════════════════
 FORMATTER SWEEP  (input: 465 words)
═════════════════════════════════════════════════════════════════════════════════════════════